# Example: download a file from a minio server

We're using boto3 for the S3 API connection. 
[boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)

and xarray to read the data file
[xarray documentation](https://xarray.pydata.org/en/stable/)



In [ ]:
# imports
import xarray as xr
# the imports are not necessary, but you do need to have them installed
# so this isjust a check that they are available

## Create a connection

Here we'll pass the secrets directly from the script, but this is very bad practice. A much better way is to create a credentials and config file. This avoids eg accidental upload of secrets to git.

For IRISCC and the deltares minio server these files will look like this:
filename: `HOME/.aws/credentials`
```toml
[default]
aws_access_key_id = YOUR_ACCESS_KEY
aws_secret_access_key = YOUR_SECRET_KEY
``` 
filename: `HOME/.aws/config`
```toml
[default]
region=eu-west-1
output=json
```

[docu on configuration](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/quickstart.html#configuration)

In [ ]:
# Connect xarray to minio zarr file
bucket_name = "iriscc"  # name of your bucket, ie the top-level folder
file_path_in_bucket = (
    "emodnet2022_depth_msl.zarr"  # path to your zarr file in the bucket
)

ds = xr.open_zarr(
    f"s3://{bucket_name}/{file_path_in_bucket}",
    storage_options={
        "key": "****",
        "secret": "*****",
        "client_kwargs": {
            "endpoint_url": "https://s3.deltares.nl",
            "region_name": "eu-west-1",
        },
        "config_kwargs": {"s3": {"addressing_style": "path"}},
    },
    consolidated=True,
)

print(ds)

In [ ]:
# select a subset of the data
ds_subset = ds.sel(lat=slice(51.5, 52.5), lon=slice(3.5, 5.0))
print(ds_subset)

In [ ]:
# plot the subset
# select the depth variable and plot with depth color scale
ds_subset.depth_msl.plot(cmap="Blues_r", vmin=-40, vmax=0)